## Model Comparison — Wildfire Risk Tier Classification

This notebook trains and evaluates multiple classifiers on the same feature matrix and train/test split used in `model_trainning.ipynb`, then compares their performance head-to-head to justify the choice of Random Forest.

**Models compared:**
1. Logistic Regression (baseline)
2. K-Nearest Neighbors
3. Decision Tree
4. Support Vector Machine (RBF kernel)
5. **Random Forest** (chosen model)
6. XGBoost
7. LightGBM

**Primary metric:** High-risk recall (class 2) — missing a high-risk tract is a safety failure.

**Secondary metrics:** F1 (macro), F1 (weighted), Accuracy, ROC-AUC (OvR macro).

## 1. Imports

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score,
    roc_auc_score, classification_report, confusion_matrix,
)

DATA_PROC = Path("../data/processed")
TIER_LABELS = {0: "Low", 1: "Medium", 2: "High"}

## 2. Load Features — Same Split as model_trainning.ipynb

In [ ]:
tracts = gpd.read_file(DATA_PROC / "features.geojson")
tracts = tracts.dropna(subset=["risk_tier"]).reset_index(drop=True)

FEATURES = [
    "dist_fire_station_m",
    "dist_hospital_m",
    "hydrant_density",
    "road_density",
    "pop_density",
    "rpl_theme_1",
    "rpl_theme_2",
    "rpl_theme_3",
    "rpl_theme_4",
]

X = tracts[FEATURES].copy()
y = tracts["risk_tier"].astype(int)

# Same seed and stratification as model_trainning.ipynb
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Total tracts : {len(X)}")
print(f"Train        : {len(X_train)}")
print(f"Test         : {len(X_test)}")
print(f"\nClass distribution (test):")
print(y_test.value_counts().sort_index().rename(TIER_LABELS))